[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/xx-quick-agents-intro.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/langchain/handbook/xx-quick-agents-intro.ipynb)

In [1]:
# Use %pip (not !pip) so packages install into THIS notebook's kernel,
# not some other Python on your system. Restart the kernel after this runs.
%pip install -qU   langchain==0.3.25   langchain-community==0.3.25   langchain-google-genai==2.0.10   sniffio anyio pyparsing

# --- OpenRouter alternative (for reference) ---
# %pip install -qU langchain-openai==0.3.22

Note: you may need to restart the kernel to use updated packages.


# Quick Agents Intro

#### [LangChain Handbook](https://pinecone.io/learn/langchain/)

In this notebook we'll see a quick intro to LangChain *agents*. For a more thorough introduction, check out [this notebook](https://github.com/pinecone-io/examples/blob/master/generation/langchain/handbook/06-langchain-agents.ipynb).

To use agents we need three main components:

* **L**arge **L**anguage **M**odels (**LLMs**)
* **Tools**
* The **Agents** themselves

Let's get started by initializing a LLM:

In [2]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or getpass("Enter your Google API key: ")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# --- OpenRouter alternative (for reference) ---
# os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
# OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_API_KEY,
    model="gemini-2.5-flash",
)

# --- OpenRouter alternative (for reference) ---
# OpenRouter is OpenAI-compatible: use the ChatOpenAI class with a different base_url.
# Pick any ":free" model from https://openrouter.ai/models, e.g. "openai/gpt-oss-120b:free".
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(
#     model="openai/gpt-oss-120b:free",
#     temperature=0,
#     api_key=OPENROUTER_API_KEY,
#     base_url="https://openrouter.ai/api/v1",
# )

D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Next, initialize a *calculator* tool using a `LLMMathChain`:

In [4]:
from langchain.chains import LLMMathChain
from langchain.agents import Tool

llm_math = LLMMathChain(llm=llm)

# initialize the math tool
math_tool = Tool(
    name='Calculator',
    func=llm_math.run,
    description='Useful for when you need to answer questions about math.'
)
# when giving tools to LLM, we must pass as list of tools
tools = [math_tool]

C:\Users\pc\AppData\Local\Temp\ipykernel_68660\2001406892.py:4: LangChainDeprecationWarning: This class is deprecated and will be removed in langchain 1.0. See API reference for replacement: https://api.python.langchain.com/en/latest/chains/langchain.chains.llm_math.base.LLMMathChain.html
  llm_math = LLMMathChain(llm=llm)
D:\Work\DEBI\Agents\.venv-langchain\Lib\site-packages\langchain\chains\llm_math\base.py:175: UserWarning: Directly instantiating an LLMMathChain with an llm is deprecated. Please instantiate with llm_chain argument or using the from_llm class method.
  warnings.warn(


In [5]:
tools[0].name, tools[0].description

('Calculator', 'Useful for when you need to answer questions about math.')

In [6]:
from langchain.agents import load_tools

tools = load_tools(
    ['llm-math'],
    llm=llm
)

In [7]:
tools[0].name, tools[0].description

('Calculator', 'Useful for when you need to answer questions about math.')

Now we use our `tools` (containing a single tool for now) to initialize an agent:

In [8]:
from langchain.agents import initialize_agent

zero_shot_agent = initialize_agent(
	agent="zero-shot-react-description",
	tools=tools,
	llm=llm,
	verbose=True,
	max_iterations=3
)

C:\Users\pc\AppData\Local\Temp\ipykernel_68660\1514443810.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  zero_shot_agent = initialize_agent(


Let's try asking some questions:

In [9]:
zero_shot_agent("what is (4.5*2.1)^2.2?")

C:\Users\pc\AppData\Local\Temp\ipykernel_68660\2247149980.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  zero_shot_agent("what is (4.5*2.1)^2.2?")




> Entering new AgentExecutor chain...


Action: Calculator
Action Input: (4.5 * 2.1)**2.2


Observation: Answer: 139.94261298333066
Thought:

I now know the final answer
Final Answer: 139.94261298333066

> Finished chain.


{'input': 'what is (4.5*2.1)^2.2?', 'output': '139.94261298333066'}

In [10]:
(4.5*2.1)**2.2

139.94261298333066

Looks good, let's try something else:

In [11]:
zero_shot_agent("if Mary has four apples and Giorgio brings two and a half apple "
                "boxes (apple box contains eight apples), how many apples do we "
                "have?")



> Entering new AgentExecutor chain...


Action: Calculator
Action Input: 2.5 * 8


Observation: Answer: 20.0
Thought:

I need to add the number of apples Mary has to the number of apples Giorgio brings.
Action: Calculator
Action Input: 4 + 20


Observation: Answer: 24
Thought:

I now know the final answer
Final Answer: We have 24 apples.

> Finished chain.


{'input': 'if Mary has four apples and Giorgio brings two and a half apple boxes (apple box contains eight apples), how many apples do we have?',
 'output': 'We have 24 apples.'}

But what if we ask a non-math question? Something that a plain and simple LLM should be able to answer easily?

In [12]:
zero_shot_agent("what is the capital of Norway?")



> Entering new AgentExecutor chain...


I now know the final answer
Final Answer: The capital of Norway is Oslo.

> Finished chain.


{'input': 'what is the capital of Norway?',
 'output': 'The capital of Norway is Oslo.'}

Unfortunately we end up hitting an error. That is because the agent is enforcing the use of a tool (this isn't always the case). A workaround we can implement is to just add another tool that can asnswer this question.

If we'd like the agent to answer this "general knowledge" question. We can, we just need to link it to an LLM tool.

We initialize the tool like so:

In [13]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

prompt = PromptTemplate(
    input_variables=["query"],
    template="{query}"
)

llm_chain = LLMChain(llm=llm, prompt=prompt)

# initialize the LLM tool
llm_tool = Tool(
    name='Language Model',
    func=llm_chain.run,
    description='use this tool for general purpose queries and logic'
)

C:\Users\pc\AppData\Local\Temp\ipykernel_68660\13945018.py:9: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=prompt)


Add the new tool to our `tools` list:

In [14]:
tools.append(llm_tool)

Now we reinitialize the agent with our two tools (`"Calculator"` and `"Language Model"`)

In [15]:
zero_shot_agent = initialize_agent(
    agent="zero-shot-react-description",
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=3
)

In [16]:
zero_shot_agent("what is the capital of Norway?")



> Entering new AgentExecutor chain...


Action: Language Model
Action Input: capital of Norway


Observation: The capital of Norway is **Oslo**.
Thought:

I now know the final answer
Final Answer: Oslo

> Finished chain.


{'input': 'what is the capital of Norway?', 'output': 'Oslo'}

In [17]:
zero_shot_agent("what is (4.5*2.1)^2.2?")



> Entering new AgentExecutor chain...


Action: Calculator
Action Input: (4.5*2.1)^2.2


Observation: Answer: 139.94261298333066
Thought:

Final Answer: 139.94261298333066

> Finished chain.


{'input': 'what is (4.5*2.1)^2.2?', 'output': '139.94261298333066'}

---